[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RadimKozl/JLNN/blob/main/examples/JLNN_nanoGPT_logical_layer.ipynb)

------------------------------------
### ***Instalation & Imports***
------------------------------------

In [ ]:
import os
import sys

In [ ]:
def setup_jlnn_custom_environment(backend: str = "gpu"):
    """
    It installs the optimal environment according to the selected backend.
    Options: "cpu", "gpu", "tpu"
    """
    backend = backend.lower().strip()
    valid_backends = ["cpu", "gpu", "tpu"]
    if backend not in valid_backends:
        raise ValueError(f"Invalid backend '{backend}'. Choose one of: {valid_backends}")

    print(f"🧹 Cleaning the environment and forcing a new installation for the backend: {backend.upper()}...")
    # We uninstall old versions to prevent driver conflicts and hidden CPU crashes
    !pip uninstall -y jax jaxlib torch --quiet

    # 1. SPECIFIC INSTALLATION OF JAXA ACCORDING TO THE CHOSEN BACKEND
    if backend == "tpu":
        print("🚀 Installing JAX with official TPU support...")
        !pip install "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html --quiet

    elif backend == "gpu":
        print("🔥 Installing JAX strictly compiled for CUDA 12+ (GPU)...")
        # Key line for stable GPU in Colab: cuda12_local from the official google releases repository
        !pip install "jax[cuda12_local]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html --quiet

    else: # cpu
        print("💻 I am installing JAX in the standard version for pure CPU calculations...")
        !pip install jax jaxlib --quiet

    # 2. SAFELY INSTALL YOUR PROJECT FROM GITHUB
    print("📦 Installing jax-lnn without the risk of overwriting downloaded CUDA drivers...")
    # The --no-deps parameter is critically important. It will prevent Pip from reading your pyproject.toml
    # and arbitrarily downloaded CPU jax/jaxlib from PyPI, which would ruin your GPU configuration.
    !pip install git+https://github.com/RadimKozl/JLNN.git --no-deps --quiet

    # 3. INSTALLING THE REST OF THE ECOSYSTEM (fixed typo in xprof, hidden logs via --quiet)
    print("📊 Installing visual, data and export tools...")
    !pip install seaborn xarray --quiet
    !pip install grain==0.2.16 orbax-checkpoint optax --quiet
    !pip install orbax-export --quiet
    !pip install pillow --quiet
    !pip install kagglehub --quiet
    !pip install tqdm --quiet
    !pip install pandas --quiet
    !pip install opencv-contrib-python-headless --quiet
    !pip install huggingface_hub --quiet
    !pip install torch tensorboard --quiet

    # Export tools (ONNX / StableHLO)
    !pip install jax2onnx onnxruntime --quiet
    !pip install tensorflow tf2onnx --quiet
    !pip install xprof --quiet
    !pip install -U tensorboard-plugin-profile --quiet
    !pip install datasets transformers --quiet

    print("\n🔄 RESTARTING COLABA CORE to apply hardware changes to Python...")
    os.kill(os.getpid(), 9)

#### ***=== AUTOMATIC TRIGGER AND ENVIRONMENT VERIFICATION ===***

💡 CHOOSE HERE WHICH BACKEND YOU WANT TO USE FOR THIS NOTEBOOK RUN ("cpu" / "gpu" / "tpu")

In [ ]:
REQUIRED_BACKEND = "gpu"

In [ ]:
try:
    import jax
    import jlnn
    import os

    # Detection and initialization, if we selected and actually caught the TPU
    if REQUIRED_BACKEND == "tpu" and 'TPU_NAME' in os.environ:
        import jax.tools.colab_tpu
        jax.tools.colab_tpu.setup_tpu()

    # Fixed check if JAX matches our wishes (safety against silent CPU crashes)
    current_platform = jax.lib.xla_bridge.get_backend().platform.lower()

    if REQUIRED_BACKEND == "gpu" and current_platform == "cpu":
        raise RuntimeError("JAX only sees the CPU, even if you demand full GPU power with CUDA 12!")

    if REQUIRED_BACKEND == "tpu" and current_platform == "cpu":
        raise RuntimeError("JAX fell into CPU mode, failed to map TPU correctly!")

    print(f"✅ The jax-lnn environment is successfully configured.")
    print(f"✅ Active hardware backend: {current_platform.upper()}")
    print(f"✅ Confirmed detected devices: {jax.devices()}")

except (ImportError, RuntimeError) as e:
    print(f"⚠️ Environment does not meet specification ({str(e)}). Running automatic repair...")
    setup_jlnn_custom_environment(backend=REQUIRED_BACKEND)

In [ ]:
!rm -rf /content/sample_data

In [ ]:
!apt install git-lfs

In [ ]:
import os, glob
import subprocess
import datetime
import shutil

import jax
from jax import export
import time
import pickle
import json
import jax.numpy as jnp
from flax import nnx
from jax.export import export
import onnx
from onnx import helper, TensorProto, numpy_helper
from typing import Any

import optax


from jax2onnx import to_onnx
import onnxruntime as ort
import jax.profiler
from jax.experimental import jax2tf

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import seaborn as sns
import tf2onnx
import tensorflow as tf
from dataclasses import dataclass

import orbax.checkpoint as ocp
import orbax.export
import grain
import grain.python as grainp
print('grain version: ', grain.__version__)
from grain import transforms as gt

from PIL import Image

# JLNN imports
from jlnn.symbolic.compiler import LNNFormula
from jlnn.nn.constraints import apply_constraints, clip_predicates
from jlnn.training.losses import total_lnn_loss
from jlnn.export.stablehlo import export_to_stablehlo, save_stablehlo_artifact
from jlnn.export.data import extract_logic_parameters
from transformers import AutoTokenizer

from huggingface_hub import (
    get_full_repo_name,
    login,
    upload_folder,
    hf_hub_download,
    snapshot_download,
    HfApi
)

from datasets import load_dataset
from torch.utils.tensorboard import SummaryWriter

from google.colab import userdata
from google.colab import files

------------------------------------
### ***Git settings***
------------------------------------

In [ ]:
def set_git_config(email, name):
    try:
        # Setting global user.email
        subprocess.run(["git", "config", "--global", "user.email", email], check=True)
        print(f"Git user.email set to: {email}")

        # Setting the global user.name
        subprocess.run(["git", "config", "--global", "user.name", name], check=True)
        print(f"Git user.name set to: {name}")

        # Check settings (optional)
        email_output = subprocess.run(["git", "config", "--global", "user.email"], capture_output=True, text=True, check=True)
        name_output = subprocess.run(["git", "config", "--global", "user.name"], capture_output=True, text=True, check=True)
        print(f"Check - Email: {email_output.stdout.strip()}")
        print(f"Check - Name: {name_output.stdout.strip()}")

    except subprocess.CalledProcessError as e:
        print(f"Error while setting up Git configuration: {e}")

In [ ]:
user_email = userdata.get('USER_EMAIL')
user_name = userdata.get('USER_NAME')
#print(user_email, user_name)
#set_git_config(user_email, user_name)

#### Loading TensorBoard extensions for Colab notebooks

In [ ]:
%load_ext tensorboard

------------------------------------
### ***Hugging Face Settings***
------------------------------------

In [ ]:
hf_token = userdata.get('HF_TOKEN') # Hugging Face token

### === ***Hugging Face Login*** ===

In [ ]:
if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    print("✅ Hugging Face login successful.")
else:
    print("❌ Error: Token 'HF_TOKEN' not found in Colab Secrets!")

In [ ]:
repo_id = "KRadim/nanogpt-cz-jlnn-punctuation"

In [ ]:
def save_to_hf(epoch):
    api = HfApi()
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

    print(f"📤 Sending checkpoints of epoch {epoch} to HF Hub...")

    epoch_path = os.path.join(os.path.abspath("checkpoints"), str(epoch))
    if not os.path.exists(epoch_path):
        print(f"⚠️ Checkpoint epoch {epoch} not found at {epoch_path}, skipping.")
        return

    for attempt in range(1, 4):
        try:
            api.upload_folder(
                folder_path=epoch_path,
                path_in_repo=f"checkpoints/{epoch}",
                repo_id=repo_id,
                repo_type="model",
                commit_message=f"Checkpoint epoch {epoch}",
                token=hf_token,
                ignore_patterns=["*.orbax-checkpoint-tmp*", "*__lock*"],
                delete_patterns=[],
            )
            print(f"✅ Done: https://huggingface.co/{repo_id}")
            break
        except Exception as e:
            print(f"⚠️ Attempt {attempt}/3 failed: {e}")
            time.sleep(5)

    try:
        api.upload_file(
            path_or_fileobj=f"logic_epoch_{epoch}.nc",
            path_in_repo=f"logs/logic_epoch_{epoch}.nc",
            repo_id=repo_id,
            token=hf_token
        )
    except Exception:
        print(f"ℹ️ Log logic_epoch_{epoch}.nc was not uploaded (may not exist).")

In [ ]:
def load_from_hf():
    # Downloads checkpoints from HF to a local folder
    snapshot_download(repo_id=repo_id, local_dir="./checkpoints")
    print("✅ Checkpoints downloaded from HF")

---------------------------------------------------
### ***Download the KRadim/czech-punctuation-pos-syntax data file***
---------------------------------------------------

#### Loading the Czech tokenizer (RobeCzech has great coverage of the syntax)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ufal/robeczech-base")
VOCAB_SIZE = tokenizer.vocab_size

---------------------------------------------------------------------------
### ***Defining transformations in Grain***

--------------------------------------------------------------------------

Grain works by defining pure Python classes (MapTransformations) that process one row from the dataset at a time.

In [ ]:
class CzechTokenizeAndPrepareTransform(grain.MapTransformation):
    def __init__(self, tokenizer, block_size: int):
        self.tokenizer = tokenizer
        self.block_size = block_size

    def map(self, element):
        # 1. Text Tokenization
        text = element.get("text", element.get("segment", ""))
        tokenized = self.tokenizer(
            text,
            max_length=self.block_size + 1,
            padding="max_length",
            truncation=True,
            return_tensors="np"
        )

        input_ids = tokenized["input_ids"][0]
        x = input_ids[:-1]
        y = input_ids[1:]

        # 2. Retrieve ALL 8 features from your enriched dataset
        has_masc = float(element.get("has_masc_animate", 0.0))
        has_subord = float(element.get("has_subord_conjunction", 0.0))
        has_comma = float(element.get("has_comma", 0.0))
        has_interrog = float(element.get("has_interrogative", 0.0))
        has_imperative = float(element.get("has_imperative", 0.0))
        has_period = float(element.get("has_period", 0.0))
        has_question = float(element.get("has_question_mark", 0.0))
        has_exclam = float(element.get("has_exclamation_mark", 0.0))

        # The array now has the correct dimension of 8, which exactly matches the model output!
        jlnn_labels = np.array([
            has_masc, has_subord, has_comma, has_interrog,
            has_imperative, has_period, has_question, has_exclam
        ], dtype=np.float32)

        return {
            "x": x,
            "y": y,
            "jlnn_labels": jlnn_labels
        }

-----------------------------------------------------------------------------
### ***Building the Grain Pipeline over the Huggingface Dataset***

-----------------------------------------------------------------------------

Now we will link the Hugging Face dataset directly to the Grain DataLoader. We will use an indexable data source (`grain.InMemoryDataSource`).

In [ ]:
print("📂 Loading splits directly from Hugging Face Hub...")
# Retrieve specific splits from the repository
train_data = load_dataset("KRadim/czech-punctuation-pos-syntax", split="train")
val_data = load_dataset("KRadim/czech-punctuation-pos-syntax", split="validation")
test_data = load_dataset("KRadim/czech-punctuation-pos-syntax", split="test")

# Create InMemoryDataSource for Grain
train_source = grain.InMemoryDataSource(train_data)
val_source = grain.InMemoryDataSource(val_data)
test_source = grain.InMemoryDataSource(test_data)

block_size = 256
batch_size = 64

transformations = [
    CzechTokenizeAndPrepareTransform(tokenizer, block_size=block_size),
    grain.Batch(batch_size=batch_size, drop_remainder=True)
]

# Sampler definition (we shuffle strictly only training data)
train_sampler = grain.IndexSampler(
    num_records=len(train_source), shuffle=True, seed=42, num_epochs=1
)
val_sampler = grain.IndexSampler(
    num_records=len(val_source), shuffle=False, seed=42, num_epochs=1
)
test_sampler = grain.IndexSampler(
    num_records=len(test_source), shuffle=False, seed=42, num_epochs=1
)

# Build the final DataLoaders
train_loader = grain.DataLoader(
    data_source=train_source, sampler=train_sampler, transformations=transformations, worker_count=4
)
val_loader = grain.DataLoader(
    data_source=val_source, sampler=val_sampler, transformations=transformations, worker_count=4
)
test_loader = grain.DataLoader(
    data_source=test_source, sampler=test_sampler, transformations=transformations, worker_count=4
)

-----------------------------------------------------------------------
### ***Model configuration***

-----------------------------------------------------------------------

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 256      # seq_len / popup window
    vocab_size: int = 10000    # The size of your Czech dictionary
    n_layer: int = 6           # Number of transformer blocks
    n_head: int = 6            # Number of attention heads
    n_embd: int = 384          # Embedding dimension (d_model)
    dropout: float = 0.1

-----------------------------------------------------------------------------
### ***Causal Self-Attention in Flax NNX***

-----------------------------------------------------------------------------

In NNX, we need to initialize the linear projections for $Q, K, V$ with the key from `nnx.Rngs`. Causal masking (so that the model does not look into the future) is solved via `jnp.tril`.

In [ ]:
class CausalSelfAttention(nnx.Module):
    def __init__(self, config: GPTConfig, rngs: nnx.Rngs):
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.block_size = config.block_size

        # Key, query, value combined into one big projection (exactly like Karpathy)
        self.c_attn = nnx.Linear(config.n_embd, 3 * config.n_embd, rngs=rngs)
        # Output projection
        self.c_proj = nnx.Linear(config.n_embd, config.n_embd, rngs=rngs)

    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        B, T, C = x.shape

        # Calculate q, k, v for all heads in the batch and move the head dimensions forward
        qkv = self.c_attn(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        # Reshape for Multi-Head: (B, T, n_head, head_size) -> Transpose to (B, n_head, T, head_size)
        q = q.reshape(B, T, self.n_head, C // self.n_head).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, self.n_head, C // self.n_head).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, self.n_head, C // self.n_head).transpose(0, 2, 1, 3)

        # Causal Attention / Dot-Product
        att = (q @ k.transpose(0, 1, 3, 2)) * (1.0 / jnp.sqrt(k.shape[-1]))

        # Applying a causal mask from below (lower triangular matrix)
        mask = jnp.tril(jnp.ones((T, T)))
        att = jnp.where(mask == 0, -jnp.inf, att)

        att = jax.nn.softmax(att, axis=-1)
        if train:
            # In NNX we can explicitly work with dropout if we want,
            # for simplicity and mathematical clarity we will omit it here
            pass

        y = att @ v # (B, n_head, T, T) x (B, n_head, T, head_size) -> (B, n_head, T, head_size)
        y = y.transpose(0, 2, 1, 3).reshape(B, T, C) # Joining the heads back together

        # Output projection
        y = self.c_proj(y)
        return y

-----------------------------------------------------------------------------
### ***Attention Pooling Head in Flax NNX***

-----------------------------------------------------------------------------

In [ ]:
class AttentionPoolingHead(nnx.Module):
    def __init__(self, d_model: int, num_classes: int, rngs: nnx.Rngs):
        # Trainable context vector (Query) registered for the optimizer
        self.query = nnx.Param(jax.random.normal(rngs.params(), (1, d_model)) * 0.02)
        self.linear = nnx.Linear(d_model, num_classes, rngs=rngs)

    def __call__(self, x: jnp.ndarray, pad_mask: jnp.ndarray = None) -> jnp.ndarray:
        # x: (B, T, d_model)
        # 1. We calculate the attention score for each token
        attn_scores = jnp.matmul(x, self.query.value.T) # (B, T, 1)

        # 2. PADDING MASK: We ignore [PAD] tokens by setting them to -inf.
        # Softmax then makes them pure zeros.
        if pad_mask is not None:
            attn_scores = jnp.where(pad_mask, attn_scores, -1e9)

        attn_weights = jax.nn.softmax(attn_scores, axis=1) # (B, T, 1)

        # 3. Weighted sum of context
        context_vector = jnp.sum(x * attn_weights, axis=1) # (B, d_model)
        return self.linear(context_vector)

-------------------------------------------------------------
### ***MLP (FeedForward) & Transformer Block***

--------------------------------------------------------------

The MLP block uses GELU activation exactly according to the GPT-2 standard.

In [ ]:
class MLP(nnx.Module):
    def __init__(self, config: GPTConfig, rngs: nnx.Rngs):
        self.c_fc = nnx.Linear(config.n_embd, 4 * config.n_embd, rngs=rngs)
        self.c_proj = nnx.Linear(4 * config.n_embd, config.n_embd, rngs=rngs)

    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        x = self.c_fc(x)
        x = jax.nn.gelu(x, approximate=True)
        x = self.c_proj(x)
        return x

In [ ]:
class Block(nnx.Module):
    def __init__(self, config: GPTConfig, rngs: nnx.Rngs):
        self.ln_1 = nnx.LayerNorm(config.n_embd, rngs=rngs)
        self.attn = CausalSelfAttention(config, rngs=rngs)
        self.ln_2 = nnx.LayerNorm(config.n_embd, rngs=rngs)
        self.mlp = MLP(config, rngs=rngs)

    def __call__(self, x: jnp.ndarray, train: bool = True) -> jnp.ndarray:
        # Correct deep-writing of the train flag into submodules
        x = x + self.attn(self.ln_1(x), train=train)
        x = x + self.mlp(self.ln_2(x), train=train)
        return x

-----------------------------------------------------------
### ***The entire JLNN nanoGPT Model***

-----------------------------------------------------------

Now we will build the complete network. We will add positional embeddings (`wpe`), token embeddings (`wte`) and most importantly a JLNN layer (`jlnn_head`) which maps the internal state of the entire text to the truth fuzzy evaluation of our 3 predicates.

In [ ]:
class JLNN_nanoGPT(nnx.Module):
    def __init__(self, config: GPTConfig, rngs: nnx.Rngs):
        self.config = config

        # Token embeddings
        self.wte = nnx.Embed(config.vocab_size, config.n_embd, rngs=rngs)
        # Positional embeddings are now a fully-fledged trainable layer
        self.wpe = nnx.Embed(config.block_size, config.n_embd, rngs=rngs)

        # Transformer blocks
        self.blocks = [Block(config, rngs=rngs) for _ in range(config.n_layer)]
        self.ln_f = nnx.LayerNorm(config.n_embd, rngs=rngs)

        # Output heads
        self.lm_head = nnx.Linear(config.n_embd, config.vocab_size, bias=False, rngs=rngs)
        # Replace weak averaging with robust Attention Pooling Head
        self.jlnn_pooling_head = AttentionPoolingHead(config.n_embd, 8, rngs=rngs)

    def __call__(self, idx: jnp.ndarray, train: bool = True) -> tuple[jnp.ndarray, jnp.ndarray]:
        B, T = idx.shape
        assert T <= self.config.block_size, f"Sequence of length {T} exceeds the limit {self.config.block_size}"

        # Making a mask for padding (we assume that ID 0 is a token [PAD])
        pad_mask = (idx != 0)[:, :, jnp.newaxis]

        # Embedding
        tok_emb = self.wte(idx)
        pos_ids = jnp.arange(0, T, dtype=jnp.int32)[jnp.newaxis, :]
        pos_emb = self.wpe(pos_ids)

        x = tok_emb + pos_emb # Broadcasting positions across the entire batch

        # Transformer passage
        for block in self.blocks:
            x = block(x, train=train)
        x = self.ln_f(x)

        # Head A: Language logits
        lm_logits = self.lm_head(x)

        # Head B: Logical predicates with intelligent pooling masking PAD tokens
        predicate_logits = self.jlnn_pooling_head(x, pad_mask=pad_mask)
        predicates = jax.nn.sigmoid(predicate_logits)

        return lm_logits, predicates

------------------------------------------------------------
### ***JLNN Loss Function with Łukasiewicz t-norm***

------------------------------------------------------------

To make the model obey your logical axiom $(Masc \land Subord) \implies Comma$, we define an exact differentiable loss for your JLNN approach.

In [ ]:
def jlnn_loss_fn(model, x_batch, y_batch, predicates_gold, lambda_logic):
    lm_logits, predicates_pred = model(x_batch)

    # ==========================================
    # 1. LANGUAGE LOSS (Language Modeling)
    # ==========================================
    B, T, V = lm_logits.shape
    log_probs = jax.nn.log_softmax(lm_logits.reshape(-1, V), axis=-1)
    one_hot = jax.nn.one_hot(y_batch.reshape(-1), num_classes=V)
    lm_loss = -jnp.mean(jnp.sum(one_hot * log_probs, axis=-1))

    # ========================================================
    # 2. PREDICATE LOSS (Supervised MSE for all 8 features)
    # ========================================================
    # Now both matrices are (B, 8), so there will be no dimensionality drop!
    pred_loss = jnp.mean((predicates_pred - predicates_gold) ** 2)

    # ==========================================
    # 3. EXTENDED LOGICAL LOSS (Fuzzy Axiomy)
    # ==========================================
    pred_Masc            = predicates_pred[:, 0]
    pred_Subord          = predicates_pred[:, 1]
    pred_Comma           = predicates_pred[:, 2]
    pred_Interrog        = predicates_pred[:, 3]
    pred_Imperative      = predicates_pred[:, 4]
    pred_Period          = predicates_pred[:, 5]
    pred_QuestionMark    = predicates_pred[:, 6]
    pred_ExclamationMark = predicates_pred[:, 7]

    # --- AXIOM 1: (Masc AND Subord) -> Comma ---
    antecedent_1 = jnp.maximum(0.0, pred_Masc + pred_Subord - 1.0)
    impl_1 = jnp.minimum(1.0, 1.0 - antecedent_1 + pred_Comma)
    loss_axiom_1 = jnp.mean(1.0 - impl_1)

    # --- AXIOM 2: Interrog -> QuestionMark ---
    impl_2 = jnp.minimum(1.0, 1.0 - pred_Interrog + pred_QuestionMark)
    loss_axiom_2 = jnp.mean(1.0 - impl_2)

    # --- AXIOM 3: Imperative -> Exclamation Mark ---
    impl_3 = jnp.minimum(1.0, 1.0 - pred_Imperative + pred_ExclamationMark)
    loss_axiom_3 = jnp.mean(1.0 - impl_3)

    # --- AXIOM 4: Exclusion (? and . cannot be valid at the same time)
    loss_axiom_4 = jnp.mean(jnp.maximum(0.0, pred_QuestionMark + pred_Period - 1.0))

    logic_loss = (loss_axiom_1 + loss_axiom_2 + loss_axiom_3 + loss_axiom_4) / 4.0

    # Combined loss
    total_loss = lm_loss + 0.2 * pred_loss + lambda_logic * logic_loss

    metrics = {
        "total": total_loss,
        "lm": lm_loss,
        "pred": pred_loss,
        "logic_total": logic_loss,
        "loss_ax_comma": loss_axiom_1,
        "loss_ax_question": loss_axiom_2,
        "loss_ax_exclamation": loss_axiom_3,
        "loss_ax_exclusion": loss_axiom_4
    }

    return total_loss, metrics

-----------------------------------------------------------------
### ***JAX Optimizer & Train Step***

-----------------------------------------------------------------

Combining Flax NNX with the `optax` library (a standard for JAX) gives us full control over updating the weights. We will use the `nnx.value_and_grad` function, which calculates the gradient backscatter in a flash.

MODEL AND OPTIMIZER INITIALIZATION (Correct NNX Setup)

In [ ]:
config = GPTConfig(vocab_size=VOCAB_SIZE, block_size=block_size)
rngs = nnx.Rngs(42)

In NNX, we initialize both the model and the optimizer as stable objects

In [ ]:
model = JLNN_nanoGPT(config, rngs=rngs)

In [ ]:
DEMO_MODE = True

In [ ]:
num_epochs = 2 if DEMO_MODE else 20
steps_per_epoch = 200 if DEMO_MODE else 1000

In [ ]:
total_steps  = num_epochs * steps_per_epoch
warmup_steps = 100 if DEMO_MODE else max(1000, total_steps // 10)

In [ ]:
print(f"total_steps:  {total_steps}")
print(f"warmup_steps: {warmup_steps}")

In [ ]:
lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=1e-4,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=1e-6
)

In [ ]:
optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(lr_schedule, weight_decay=0.01)
    ),
    wrt=nnx.Param
)

------------------------------------------------------------
### ***SAFE NNX JIT TRAIN STEP***
------------------------------------------------------------

In NNX we use jann.jit or @nnx.jit directly above the function that internally modifies the objects it has in the closure.
NNX takes care of the correct unpacking and packing state (Functional API) for pure JAX.

In [ ]:
# Initialize native Orbax Checkpointer for Flax NNX graphs
checkpointer = nnx.DirCheckpointer(
    directory=os.path.abspath("checkpoints"),
    save_args=dict(model=model, optimizer=optimizer)
)

In [ ]:
@nnx.jit
def train_step(x_batch, y_batch, predicates_gold, lambda_logic):
    # has_aux=True ensures that the monitoring dictionary is passed out without affecting the gradients
    (loss_val, metrics), grads = nnx.value_and_grad(jlnn_loss_fn, has_aux=True)(
        model, x_batch, y_batch, predicates_gold, lambda_logic
    )
    optimizer.update(grads)
    return metrics

In [ ]:
@nnx.jit
def eval_step(x_batch, y_batch, predicates_gold, lambda_logic):
    # jlnn_loss_fn returns a pair: total_loss (for grads) and a metrics dictionary
    loss_val, metrics = jlnn_loss_fn(model, x_batch, y_batch, predicates_gold, lambda_logic)

    # We return the metrics dictionary so we can pull "total" out of it
    return metrics

-----------------------------------------------------------------------
### ***CORRECT TRAINING LOOP WITH GRAIN DATA-LOADER***

-----------------------------------------------------------------------

#### Initializing the TensorBoard logging directory

In [ ]:
log_dir = os.path.abspath("tb_logs")
writer = SummaryWriter(log_dir=log_dir)

In [ ]:
total_train_batches = len(train_loader)
print(f"🚀 Starting JLNN nanoGPT training. Epochs: {num_epochs}")

In [ ]:
global_step = 0

In [ ]:
for epoch in range(1, num_epochs + 1):
    # --- TRAINING PHASE ---
    print(f"\n📺 Epoch {epoch}/{num_epochs} - Training")
    epoch_train_loss = 0.0

    for step, batch in enumerate(tqdm(train_loader, desc="Training")):
        x_batch = jnp.array(batch["x"])
        y_batch = jnp.array(batch["y"])
        jlnn_gold = jnp.array(batch["jlnn_labels"])

        # Linear annealing of logical axiom weights
        progress = step / total_train_batches
        if progress < 0.2:
            lambda_logic = jnp.array(0.0, dtype=jnp.float32)
        else:
            val = (progress - 0.2) / 0.5
            lambda_logic = jnp.array(min(1.0, val), dtype=jnp.float32)

        # ==================== HARDWARE PROFILER (XProf) ====================
        # We profile up to the 2nd epoch so that the initial JIT compilation from the 1st epoch does not distort the real performance
        if epoch == 2 and step == 5:
            print("\n📸 Starting XProf hardware trace (capturing hardware bottlenecks)...")
            jax.profiler.start_trace(log_dir)

        loss = train_step(x_batch, y_batch, jlnn_gold, lambda_logic)
        epoch_train_loss += loss["total"].item()

        if epoch == 2 and step == 15:
            # block_until_ready() ensures that asynchronous JAX waits for operations to complete on the GPU/TPU
            y_batch.block_until_ready()
            jax.profiler.stop_trace()
            print("📸 XProf hardware trace finished and saved.")
        # ===================================================================

        # Writing training metrics to TensorBoard
        global_step += 1
        writer.add_scalar("Loss/Train/Total", loss["total"].item(), global_step)
        writer.add_scalar("Loss/Train/Language_Modeling", loss["lm"].item(), global_step)
        writer.add_scalar("Loss/Train/Predicate_MSE", loss["pred"].item(), global_step)
        writer.add_scalar("Loss/Train/Logic_Total", loss["logic_total"].item(), global_step)
        writer.add_scalar("Axioms/Train/Axiom_1_Comma", loss["loss_ax_comma"].item(), global_step)
        writer.add_scalar("Axioms/Train/Axiom_2_Question", loss["loss_ax_question"].item(), global_step)
        writer.add_scalar("Axioms/Train/Axiom_3_Exclamation", loss["loss_ax_exclamation"].item(), global_step)
        writer.add_scalar("Axioms/Train/Axiom_4_Exclusion", loss["loss_ax_exclusion"].item(), global_step)

    avg_train_loss = epoch_train_loss / total_train_batches

    # --- VALIDATION PHASE ---
    print(f"🧪 Epoch {epoch}/{num_epochs} - Validation")
    epoch_val_loss = 0.0
    total_val_batches = len(val_loader)

    # Fix lambda to 1.0 as a JAX field for validation
    lambda_val_jax = jnp.array(1.0, dtype=jnp.float32)

    for batch in tqdm(val_loader, desc="Validace"):
        x_batch = jnp.array(batch["x"])
        y_batch = jnp.array(batch["y"])
        jlnn_gold = jnp.array(batch["jlnn_labels"])

        # During validation, we want to strictly penalize violations of the t-norm (lambda=1.0)
        val_metrics = eval_step(x_batch, y_batch, jlnn_gold, lambda_logic=lambda_val_jax)
        epoch_val_loss += val_metrics["total"].item()

    avg_val_loss = epoch_val_loss / total_val_batches

   # Writing validation metrics at the end of the epoch
    writer.add_scalar("Loss/Validation/Total", avg_val_loss, epoch)

    print(f"📊 [Epoch {epoch}] Results -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- AUTOMATIC CHECKPOINT ---
    checkpointer.save(step=epoch, model=model, optimizer=optimizer)
    print(f"💾 Checkpoint for epoch {epoch} successfully saved to disk.")

    # Optionally send to HF hub if needed:
    save_to_hf(epoch)

writer.close()

--------------------------------------------------------
### ***Viewing TensorBoard and XProf Bottlenecks***

--------------------------------------------------------

Running this cell will open the integrated TensorBoard interface directly in Colab.
In the top menu, you can switch the tab from "Scalars" to "**Profile**", where you will see a complete
XProf analysis of the hardware (including operation breakdowns, Kernels and any bottlenecks caused by memory or host).

Forcing TensorBoard to appear above the metrics and profiler logs folder

In [ ]:
%tensorboard --logdir ./tb_logs

-----------------------------------------------------------------------------
### ***Pure MLIR and StableHLO Export Layer***

-----------------------------------------------------------------------------

JAX stores computational graphs natively in MLIR modules, which internally use the StableHLO dialect. To take full advantage of the compiled StableHLO format (which requires fixed matrix dimensions), we export the model in two variants:

1. For testing on the entire `test_loader` in its native batch form (64, 256).
2. For fast inference of custom text strings in its lightweight form (1, 256).

We will save the model in both MLIR text format (human-readable ASM instructions) and in optimized binary bytecode (MLIRBC) for both dimensions and at the same time create a production StableHLO artifact using the JLNN utility.

In [ ]:
print("📦 Lowering Python graph directly to native MLIR and StableHLO representations...")

#### Pure function required by JAX export (works with state isolated outside the model)

In [ ]:
def forward_pure(params, graphdef, idx):
    model_reconstructed = nnx.merge(graphdef, params)
    return model_reconstructed(idx, train=False)

#### Dividing the NNX model into static structure and dynamic parameters

In [ ]:
graphdef, params = nnx.split(model)

#### Definition of abstract dimensions as inputs to the compiler

In [ ]:
abstract_batch_idx = jnp.ones((64, 256), dtype=jnp.int32)
abstract_single_idx = jnp.ones((1, 256), dtype=jnp.int32)

#### Running native JAX export to MLIR environment

In [ ]:
exported_mlir_batch = jax.export.export(
    jax.jit(lambda idx: forward_pure(params, graphdef, idx))
)(abstract_batch_idx)

In [ ]:
exported_mlir_single = jax.export.export(
    jax.jit(lambda idx: forward_pure(params, graphdef, idx))
)(abstract_single_idx)

#### Create target folders for MLIR artifacts

In [ ]:
os.makedirs("artifacts/mlir_batch_64", exist_ok=True)
os.makedirs("artifacts/mlir_single_1", exist_ok=True)

#### 1. SAVE BATCH VERSION (64, 256)

Binary MLIR Bytecode (MLIRBC) - ideal for fast loading by production runtimes



In [ ]:
with open("artifacts/mlir_batch_64/model.mlirbc", "wb") as f:
    f.write(exported_mlir_batch.mlir_module_serialized)

#### Human-readable text MLIR code (in ASM text syntax of the StableHLO dialect)

In [ ]:
with open("artifacts/mlir_batch_64/model.mlir", "w") as f:
    f.write(exported_mlir_batch.mlir_module_text)

#### 2. SAVE SINGLE INFERENCE VERSION (1, 256)

In [ ]:
with open("artifacts/mlir_single_1/model.mlirbc", "wb") as f:
    f.write(exported_mlir_single.mlir_module_serialized)

In [ ]:
with open("artifacts/mlir_single_1/model.mlir", "w") as f:
    f.write(exported_mlir_single.mlir_module_text)

#### Saving the original production artifact using the jlnn utility

In [ ]:
stablehlo_dir = "./stablehlo_model"
os.makedirs(stablehlo_dir, exist_ok=True)
save_stablehlo_artifact(exported_mlir_single, os.path.join(stablehlo_dir, "model.stablehlo"))

In [ ]:
print("✅ Native MLIR text modules (.mlir), bytecodes (.mlirbc) and StableHLO artifact successfully saved.")

--------------------------------------------------------------------------
### ***📤 SAVING EXPORTED PRODUCTION MODELS ON THE HUGGING FACE HUB***

--------------------------------------------------------------------------

In [ ]:
print("📤 Uploading production MLIR and StableHLO artifacts to Hugging Face Hub...")

for attempt in range(1, 4):
    try:
        api = HfApi()

        # 1. Upload folder with MLIR modules (.mlir and .mlirbc for both dimensions)
        api.upload_folder(
            folder_path=os.path.abspath("artifacts"),
            path_in_repo="artifacts",
            repo_id=repo_id,
            repo_type="model",
            commit_message="Upload compiled MLIR text and bytecode models",
            token=hf_token,
        )

        # 2. Upload folder with StableHLO production artifact
        api.upload_folder(
            folder_path=os.path.abspath(stablehlo_dir),
            path_in_repo="stablehlo_model",
            repo_id=repo_id,
            repo_type="model",
            commit_message="Upload compiled StableHLO model artifact",
            token=hf_token,
        )

        print(f"🚀 All compilation artifacts successfully uploaded to: https://huggingface.co/{repo_id}")
        break
    except Exception as e:
        print(f"⚠️ Upload attempt {attempt}/3 failed: {e}")
        if attempt < 3:
            time.sleep(5)

### ***TESTING ON OFFICIAL TEST DATASETS***

In [ ]:
print("\n🧮 Running evaluation of compiled StableHLO model on test data...")
stablehlo_batch_fn = exported_mlir_batch.call
test_loss_cum = 0.0
total_test_batches = len(test_loader)

In [ ]:
for batch in tqdm(test_loader, desc="StableHLO testing"):
    x_batch = jnp.array(batch["x"])
    y_batch = jnp.array(batch["y"])
    jlnn_gold = jnp.array(batch["jlnn_labels"])

    # Run inference over a fast StableHLO graph
    lm_logits, predicates_pred = stablehlo_batch_fn(x_batch)

    # Calculating the total loss on the test data
    val_loss = eval_step(x_batch, y_batch, jlnn_gold, lambda_logic=1.0)
    test_loss_cum += val_loss.item()

In [ ]:
print(f"🏆 Resulting Test Loss of compiled model: {test_loss_cum / total_test_batches:.4f}")

### ***--- INFERENCE ON OWN TEXT STRINGS ---***

In [ ]:
stablehlo_single_fn = exported_mlir_single.call

In [ ]:
def prepare_custom_text(text: str, tokenizer, block_size: int):
    tokenized = tokenizer(
        text,
        max_length=block_size,
        padding="max_length",
        truncation=True,
        return_tensors="np"
    )
    return jnp.array(tokenized["input_ids"], dtype=jnp.int32)

In [ ]:
test_sentence = [
    "Mladý muž, protože spěchal na vlak, běžel jako o závod.",  # Comma + Period
    "Kočka dneska spala na okně.",                              # Comma=0, Period=1
    "Proč jsi včera nepřišel na domluvenou schůzku?",           # Question mark
    "Okamžitě se vrať zpátky do svého pokoje!",                 # Exclamation mark
]

In [ ]:
print("\n🔥 Testing inference on own sentences (StableHLO):")
print("-" * 65)

In [ ]:
for sentence in test_sentence:
    input_tokens = prepare_custom_text(sentence, tokenizer, block_size) # Corrected from a sentence to a sentence
    lm_logits, predicates = stablehlo_single_fn(input_tokens)
    preds = predicates[0]  # Removing batch size (1, 8) -> (8,)

    print(f"\nSentence: \"{sentence}\"")
    print(f"  -> [Masc] Life Masculine:               {preds[0]:.4f}")
    print(f"  -> [Subord] Subordinate clutch:         {preds[1]:.4f}")
    print(f"  -> [Comma] Comma (,):                   {preds[2]:.4f}")
    print(f"  -> [Interrog] Interrogative element:    {preds[3]:.4f}")
    print(f"  -> [Imperative] Command:                {preds[4]:.4f}")
    print(f"  -> [Period] Period (.):                 {preds[5]:.4f}")
    print(f"  -> [Question] Question mark (?):        {preds[6]:.4f}")
    print(f"  -> [Exclamation] Exclamation mark (!):  {preds[7]:.4f}")

    # Mathematical verification of axioms in NumPy
    ax1_antecedent = max(0.0, float(preds[0] + preds[1] - 1.0))
    ax1_truth = min(1.0, 1.0 - ax1_antecedent + float(preds[2]))

    ax2_truth = min(1.0, 1.0 - float(preds[3]) + float(preds[6]))
    ax3_truth = min(1.0, 1.0 - float(preds[4]) + float(preds[7]))
    ax4_violation = max(0.0, float(preds[6] + preds[5] - 1.0))

    print(f"    📐 Truth of Axiom 1 ((Masc ∧ Subord) -> ,):             {ax1_truth:.4f}")
    print(f"    📐 The truth of Axiom 2 (Interrog -> ?):                {ax2_truth:.4f}")
    print(f"    📐 Truth of Axiom 3 (Imperative -> !):                  {ax3_truth:.4f}")
    print(f"    📐 Violation of Axiom 4 (Concurrence of ? and .):       {ax4_violation:.4f} (we want 0.0)")